 ##Gold Aggregation Tables for Genie
 
 **agentic-civic-resolution-app**

 **Goal:** Build pre-aggregated Gold tables that make Genie faster and more accurate.
 Genie works best when tables are clean, narrow, and well-described.

 | Table | Powers |
 |-------|--------|
 | `civicops.gold.complaints_by_borough_month` | Ward/area trend queries |
 | `civicops.gold.dept_sla_performance` | Department performance queries |
 | `civicops.gold.complaint_type_trends` | "Why is X increasing" queries |
 | `civicops.gold.severity_summary` | Critical complaint dashboards |

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS civicops.gold")

## 1. Complaints by Borough & Month

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE civicops.gold.complaints_by_borough_month AS
SELECT
    borough_clean                                                   AS borough,
    open_year                                                       AS year,
    open_month                                                      AS month,
    MAKE_DATE(open_year, open_month, 1)                             AS period_start,
    complaint_type,
    COUNT(*)                                                        AS total_complaints,
    SUM(CASE WHEN is_closed = FALSE THEN 1 ELSE 0 END)             AS unresolved,
    SUM(CASE WHEN is_closed = TRUE  THEN 1 ELSE 0 END)             AS resolved,
    ROUND(AVG(days_open), 1)                                        AS avg_days_open,
    ROUND(
        SUM(CASE WHEN is_closed = FALSE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1
    )                                                               AS unresolved_pct
FROM civicops.silver.complaints
WHERE borough_clean IS NOT NULL
  AND open_year IS NOT NULL
  AND open_month IS NOT NULL
GROUP BY borough_clean, open_year, open_month, complaint_type
""")

count = spark.table("civicops.gold.complaints_by_borough_month").count()
print(f"✓ complaints_by_borough_month: {count:,} rows")

In [0]:
%sql
desc civicops.gold.processed_tickets

## 2. Department SLA Performance

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE civicops.gold.dept_sla_performance AS
SELECT
    t.dept_code,
    t.dept_name,
    t.category,
    COUNT(*)                                                AS total_tickets,
    SUM(CASE WHEN t.escalate = TRUE THEN 1 ELSE 0 END)     AS escalated,
    SUM(CASE WHEN t.health_risk = TRUE THEN 1 ELSE 0 END)  AS health_risk_count,
    ROUND(AVG(t.sla_hours), 0)                              AS avg_sla_hours,
    MIN(t.sla_hours)                                        AS min_sla_hours,
    MAX(t.sla_hours)                                        AS max_sla_hours,
    ROUND(AVG(t.processing_ms) / 1000.0, 2)                AS avg_processing_sec,
    COUNT(DISTINCT t.officer_tier)                          AS officer_tiers_used,
    MAX(t.created_at)                                       AS last_ticket_at
FROM civicops.gold.processed_tickets t
GROUP BY t.dept_code, t.dept_name, t.category
""")

count = spark.table("civicops.gold.dept_sla_performance").count()
print(f"✓ dept_sla_performance: {count:,} rows")

## 3. Complaint Type Trends (Monthly)

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE civicops.gold.complaint_type_trends AS
SELECT
    complaint_type,
    open_year                                                   AS year,
    open_month                                                  AS month,
    MAKE_DATE(open_year, open_month, 1)                         AS period_start,
    COUNT(*)                                                    AS complaint_count,
    ROUND(AVG(days_open), 1)                                    AS avg_days_open,
    SUM(CASE WHEN is_closed = FALSE THEN 1 ELSE 0 END)         AS unresolved_count,
    -- Month-over-month change (requires window function)
    COUNT(*) - LAG(COUNT(*)) OVER (
        PARTITION BY complaint_type
        ORDER BY open_year, open_month
    )                                                           AS mom_change,
    ROUND(
        (COUNT(*) - LAG(COUNT(*)) OVER (
            PARTITION BY complaint_type ORDER BY open_year, open_month
        )) * 100.0 / NULLIF(LAG(COUNT(*)) OVER (
            PARTITION BY complaint_type ORDER BY open_year, open_month
        ), 0), 1
    )                                                           AS mom_pct_change
FROM civicops.silver.complaints
WHERE open_year IS NOT NULL AND open_month IS NOT NULL
GROUP BY complaint_type, open_year, open_month
""")

count = spark.table("civicops.gold.complaint_type_trends").count()
print(f"✓ complaint_type_trends: {count:,} rows")


## 4. Severity Summary (Live View)

In [0]:

spark.sql("""
CREATE OR REPLACE TABLE civicops.gold.severity_summary AS
SELECT
    severity_score,
    priority_label,
    category,
    dept_code,
    dept_name,
    COUNT(*)                                                        AS total,
    SUM(CASE WHEN escalate = TRUE THEN 1 ELSE 0 END)               AS escalated,
    SUM(CASE WHEN health_risk = TRUE THEN 1 ELSE 0 END)            AS health_risk_count,
    SUM(CASE WHEN infrastructure_risk = TRUE THEN 1 ELSE 0 END)    AS infra_risk_count,
    ROUND(AVG(sla_hours), 0)                                        AS avg_sla_hours,
    ROUND(AVG(processing_ms) / 1000.0, 2)                          AS avg_processing_sec
FROM civicops.gold.processed_tickets
GROUP BY severity_score, priority_label, category, dept_code, dept_name
""")

count = spark.table("civicops.gold.severity_summary").count()
print(f"✓ severity_summary: {count:,} rows")

## 5. Verify All Gold Tables

In [0]:
gold_tables = [
    "civicops.gold.processed_tickets",
    "civicops.gold.complaints_by_borough_month",
    "civicops.gold.dept_sla_performance",
    "civicops.gold.complaint_type_trends",
    "civicops.gold.severity_summary",
]

print(f"{'Table':<45} {'Rows':>10}")
print("-" * 57)
for t in gold_tables:
    try:
        n = spark.table(t).count()
        print(f"{t:<45} {n:>10,}")
    except Exception as e:
        print(f"{t:<45} ERROR: {e}")